# Mooring Field Detection — Kaggle GPU Pipeline

**Before running:**
1. Settings → Accelerator → **GPU T4** (not P100)
2. Settings → Internet → **On**
3. Add-ons → Secrets → add `GOOGLE_MAPS_API_KEY`

Run cells top to bottom. Do **not** restart the kernel mid-session.

In [ ]:
# Cell 1 — Clone repo and install dependencies
import subprocess, sys

!git clone https://github.com/YOUR_USER/MooringFieldDetection.git /kaggle/working/MooringFieldDetection
%cd /kaggle/working/MooringFieldDetection

# Install packages missing from Kaggle's environment
for pkg in ["ultralytics", "httpx", "python-dotenv"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True)

# Register mooring_fields package without touching Kaggle's pre-installed libs
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], capture_output=True)
print("install done")

In [ ]:
# Cell 2 — Fix Python path (required: editable install .pth files aren't read by running kernel)
import sys
sys.path.insert(0, "/kaggle/working/MooringFieldDetection/src")

import mooring_fields
print("mooring_fields loaded from:", mooring_fields.__file__)

In [ ]:
# Cell 3 — Load API key from Kaggle Secrets and fetch satellite imagery
import os, json
from kaggle_secrets import UserSecretsClient
os.environ["GOOGLE_MAPS_API_KEY"] = UserSecretsClient().get_secret("GOOGLE_MAPS_API_KEY")

from mooring_fields.fetch_imagery import fetch_all
result = fetch_all()
print(json.dumps(result, indent=2))
# Expect: downloaded ~615 tiles (or 0 if already cached)

In [ ]:
# Cell 4 — Prelabel: run YOLO inference on all tiles to generate boat annotations
import json
from mooring_fields.prelabel_boats import prelabel_all
result = prelabel_all()
print(json.dumps(result, indent=2))
# Expect: images and detections counts for train and val splits

In [ ]:
# Cell 5 — Train: fine-tune YOLOv8l-OBB on prelabeled boat annotations
# Uses config/training.yaml settings (yolov8l-obb.pt, 150 epochs, batch_gpu=8)
# Takes ~45-90 min on T4 — watch epoch logs scroll below
import json
from mooring_fields.train_boats import train
from mooring_fields.runtime import publish_outputs

report = train()
report["published"] = publish_outputs()
print(json.dumps(
    {k: v for k, v in report.items() if k != "results"},
    indent=2
))
# Key metrics: mAP50 (target >0.83), best_weights path

In [ ]:
# Cell 6 — Evaluate: detect and cluster boats on val sites, compute Hit@150m
import json
from mooring_fields.evaluate import evaluate_val
from mooring_fields.runtime import publish_outputs

report = evaluate_val()
report["published"] = publish_outputs()

# Print summary only (skip per_site detail)
summary = {k: v for k, v in report.items() if k not in ("per_site", "clusters")}
print(json.dumps(summary, indent=2))
# Key metric: hit_rate_pct — % of val mooring fields correctly detected

## Optional: scan new locations

After training, you can run detection on **any coastal location worldwide**.
Drop pins in Google Earth, export as KML, then run the cells below.

In [ ]:
# Cell 7 (optional) — Parse a new KML of candidate locations
# Upload your KML to Kaggle first, then update the path below
from mooring_fields.split_sites import run_parse_and_split
from pathlib import Path
import json

# result = run_parse_and_split(kml=Path("/kaggle/input/your-dataset/your_locations.kml"))
# print(json.dumps(result, indent=2))

In [ ]:
# Cell 8 (optional) — Fetch imagery for new locations and run detection
from mooring_fields.fetch_imagery import fetch_all
from mooring_fields.cluster_fields import run_on_split
from mooring_fields.kml_export import clusters_to_kml
from pathlib import Path
import json

# Fetch new tiles
# fetch_all(split="train")

# Detect mooring fields using trained weights
# weights = Path("/kaggle/working/MooringFieldDetection/runs/mooring_boats/weights/best.pt")
# clusters = run_on_split(split="train", weights=weights)
# print(f"Found {len(clusters)} mooring fields")
# for c in sorted(clusters, key=lambda x: -x.boat_count):
#     print(f"  {c.boat_count} boats at {c.lat:.5f}, {c.lon:.5f}")

# Export to KML — download from Output tab and open in Google Earth
# clusters_to_kml(clusters, Path("discovered_fields.kml"), "Global scan")